# 02 Preprocessing

Create PyTorch datasets, transforms, and dataloaders for image deepfake classification.

In [1]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

DATA_ROOT = Path("../data/splits/images")
SPLITS = ["train", "val", "test"]
CLASSES = ["real", "fake"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
BATCH_SIZE = 32
NUM_WORKERS = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
def collect_image_paths(data_root: Path) -> pd.DataFrame:
    rows = []
    for split in SPLITS:
        for class_name in CLASSES:
            class_dir = data_root / split / class_name
            print(f"Scanning {class_dir} | exists={class_dir.exists()}")
            if not class_dir.exists():
                continue
            for path in sorted(class_dir.iterdir()):
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append({"path": str(path), "split": split, "label": class_name, "label_id": 1 if class_name == "fake" else 0})
    return pd.DataFrame(rows)


df = collect_image_paths(DATA_ROOT)
print(f"Loaded records: {len(df)}")
df.head()

Scanning ..\data\splits\images\train\real | exists=True
Scanning ..\data\splits\images\train\fake | exists=True
Scanning ..\data\splits\images\val\real | exists=True
Scanning ..\data\splits\images\val\fake | exists=True
Scanning ..\data\splits\images\test\real | exists=True
Scanning ..\data\splits\images\test\fake | exists=True
Loaded records: 57098


,path,split,label,label_id
0,..\data\splits\images\train\real\real_100.jpg,train,real,0
1,..\data\splits\images\train\real\real_10001.jpg,train,real,0
2,..\data\splits\images\train\real\real_10002.jpg,train,real,0
3,..\data\splits\images\train\real\real_10006.jpg,train,real,0
4,..\data\splits\images\train\real\real_10013.jpg,train,real,0


In [3]:
imagenet_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


class DeepfakeImageDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        print(f"Dataset initialized with {len(self.dataframe)} images")

    def __len__(self):
        return len(self.dataframe)

    def _load_image(self, path: str):
        try:
            with Image.open(path) as image:
                return image.convert("RGB")
        except (OSError, UnidentifiedImageError) as error:
            print(f"Warning: corrupted image replaced with blank tensor: {path} | {error}")
            return Image.new("RGB", (224, 224), color=(0, 0, 0))

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = self._load_image(row["path"])
        label = torch.tensor(row["label_id"], dtype=torch.float32)

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [4]:
datasets = {
    split: DeepfakeImageDataset(df[df["split"] == split], transform=imagenet_transform)
    for split in SPLITS
}

dataloaders = {
    "train": DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
    "val": DataLoader(datasets["val"], batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
    "test": DataLoader(datasets["test"], batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
}

for split in SPLITS:
    print(f"{split} dataset size: {len(datasets[split])}")
    print(f"{split} batches: {len(dataloaders[split])}")

Dataset initialized with 42000 images
Dataset initialized with 11828 images
Dataset initialized with 3270 images
train dataset size: 42000
train batches: 1313
val dataset size: 11828
val batches: 370
test dataset size: 3270
test batches: 103


In [5]:
for split, loader in dataloaders.items():
    images, labels = next(iter(loader))
    print(f"{split} batch image shape: {images.shape}")
    print(f"{split} batch label shape: {labels.shape}")
    print(f"{split} labels: {labels[:10].tolist()}")

train batch image shape: torch.Size([32, 3, 224, 224])
train batch label shape: torch.Size([32])
train labels: [1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0]
val batch image shape: torch.Size([32, 3, 224, 224])
val batch label shape: torch.Size([32])
val labels: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
test batch image shape: torch.Size([32, 3, 224, 224])
test batch label shape: torch.Size([32])
test labels: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
